In [1]:
import requests
from bs4 import BeautifulSoup
import urllib3

# HTTPS 요청 시 SSL 인증서 검증 경고를 비활성화하여 보안 경고를 방지
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)  

In [2]:
# 1. 웹페이지 가져오기 (정적 크롤링 시도)
url = "https://topis.seoul.go.kr/map/openBusMap.do"
response = requests.get(url, verify=False)  # GET 요청 + SSL 검증 비활성화

print("문제 1. 웹페이지 가져오기 (정적 크롤링 시도)")
print("답: 상태 코드:", response.status_code)  
print("설명: status_code=200이면 HTML은 내려받았다는 뜻이며, 데이터가 포함됐다는 의미는 아닙니다.")

# HTML 파싱
soup = BeautifulSoup(response.text, 'html.parser')  


문제 1. 웹페이지 가져오기 (정적 크롤링 시도)
답: 상태 코드: 200
설명: status_code=200이면 HTML은 내려받았다는 뜻이며, 데이터가 포함됐다는 의미는 아닙니다.


In [3]:
# 2. '검색결과 컨테이너'는 존재하는가?
print("\n문제 2. 검색결과 컨테이너 존재 여부 확인")

bus_container = soup.select("#resultList")     
stn_container = soup.select("#resultList2")    

print("답: #resultList 존재:", bus_container is not None)
print("답: #resultList2 존재:", stn_container is not None)
print("설명: 컨테이너가 존재하면, '결과를 렌더링할 자리'는 HTML에 포함되어 있다는 뜻입니다.")



문제 2. 검색결과 컨테이너 존재 여부 확인
답: #resultList 존재: True
답: #resultList2 존재: True
설명: 컨테이너가 존재하면, '결과를 렌더링할 자리'는 HTML에 포함되어 있다는 뜻입니다.


In [4]:
# 3. 컨테이너 내부에 실제 데이터(li)가 들어있는가?
print("\n문제 3. 컨테이너 내부 데이터(<li>) 존재 여부 확인")

bus_items = soup.select("#resultList > li")    
stn_items = soup.select("#resultList2 > li")   

print("답: 버스 결과 li 개수:", len(bus_items))  
print("답: 정류소 결과 li 개수:", len(stn_items)) 
print("설명: 정적 HTML에서 li가 0개면, 데이터가 초기 HTML에 포함되지 않았을 가능성이 큽니다.")



문제 3. 컨테이너 내부 데이터(<li>) 존재 여부 확인
답: 버스 결과 li 개수: 0
답: 정류소 결과 li 개수: 0
설명: 정적 HTML에서 li가 0개면, 데이터가 초기 HTML에 포함되지 않았을 가능성이 큽니다.


In [6]:
# 4. (증거 수집) HTML에 '검색을 수행하는 자바스크립트 함수 호출 흔적'이 있는가?
print("\n문제 4. 동적 렌더링(자바스크립트) 흔적 찾기")

# 검색 버튼은 fn_search('1') 같은 JS 함수로 동작함
has_fn_search = "fn_search" in response.text
has_fn_search_bus_stn = "fn_searchBusStn" in response.text

print("답: fn_search 문자열 존재:", has_fn_search)
print("답: fn_searchBusStn 문자열 존재:", has_fn_search_bus_stn)

print("설명: 검색 기능이 자바스크립트 함수(fn_search 등)로 구현되어 있고, "
      "그 결과를 DOM에 채우는 구조일 가능성을 시사합니다.")


문제 4. 동적 렌더링(자바스크립트) 흔적 찾기
답: fn_search 문자열 존재: True
답: fn_searchBusStn 문자열 존재: True
설명: 검색 기능이 자바스크립트 함수(fn_search 등)로 구현되어 있고, 그 결과를 DOM에 채우는 구조일 가능성을 시사합니다.


In [7]:
# 5. 결론: 이 페이지는 정적 크롤링으로 데이터 수집이 가능한가?
print("\n문제 5. 결론 도출 (정적 크롤링 실패 원인 정리)")

if response.status_code == 200 and (bus_container or stn_container) and len(bus_items) == 0 and len(stn_items) == 0:
    print("답: 이 페이지의 '검색 결과 데이터'는 정적 크롤링(requests + BeautifulSoup)만으로 수집하기 어렵습니다.")
    print("설명: HTML에는 검색결과를 담을 컨테이너(#resultList/#resultList2)만 있고, "
          "실제 데이터(<li>)는 비어 있습니다. 이는 브라우저에서 자바스크립트가 실행되며 "
          "추가 요청(AJAX/XHR 등)으로 데이터를 받아 DOM에 채우는 '동적 렌더링' 구조일 가능성이 큽니다.")
    print("추가 안내: 해결하려면 Selenium(브라우저 실행)으로 렌더링 후 파싱하거나, "
          "개발자도구(Network)에서 XHR 요청을 찾아 requests로 직접 호출해야 합니다.")
else:
    print("답: 정적 크롤링으로도 데이터가 포함되어 있을 수 있습니다. (현재 조건에서는 단정하기 어려움)")
    print("설명: 페이지 구조/응답이 환경에 따라 달라질 수 있으므로, 컨테이너/데이터 존재 여부를 추가 확인하세요.")


문제 5. 결론 도출 (정적 크롤링 실패 원인 정리)
답: 이 페이지의 '검색 결과 데이터'는 정적 크롤링(requests + BeautifulSoup)만으로 수집하기 어렵습니다.
설명: HTML에는 검색결과를 담을 컨테이너(#resultList/#resultList2)만 있고, 실제 데이터(<li>)는 비어 있습니다. 이는 브라우저에서 자바스크립트가 실행되며 추가 요청(AJAX/XHR 등)으로 데이터를 받아 DOM에 채우는 '동적 렌더링' 구조일 가능성이 큽니다.
추가 안내: 해결하려면 Selenium(브라우저 실행)으로 렌더링 후 파싱하거나, 개발자도구(Network)에서 XHR 요청을 찾아 requests로 직접 호출해야 합니다.
